# 10 · 第一个工具：按症状给所有罕见病排序

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → [10 第一个工具 ✅] → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 起：改进 agent …
```

> **这一章的代码在合并请求 #12 里。** 这一章不调大模型，一分钱不花；每次评测十几秒。

## 背景：先做哪个工具

计划里 agent 有四个工具（第 00 章）：

| 工具 | 做什么 |
|---|---|
| T1 症状标准化 | 医生写的话 → 标准症状编号（HPO） |
| **T2 按症状给疾病排序** | 一组症状编号 → 所有罕见病按吻合程度排好 |
| T3 疾病知识 | 某个病的典型症状、遗传方式、致病基因 |
| T4 吻合度核对 | 病人的症状和某个病比：对上的、缺的、矛盾的 |

原来打算先做 T1。但我们的考题（Phenopacket 病例）给的**本来就是标准编号**，T1 现在没法用分数检验。T2 能马上见分晓，而且它不用大模型，正好就是计划里的对照组"传统方法"（B0）。所以先做 T2：做完它既是 agent 的第一个工具，又多了一个基线，能和大模型的 28% 直接比。T1 等接上自由文本的病例再做。

## 这一章做的 7 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 工具要读的知识 | HPO 官方发布 + 数据源清单 | 知识也钉版本、核对指纹 |
| 2 | 用现成的库打分 | hpo3（PyHPO 的 Rust 版） | 不自己造轮子；用行业通用的算法 |
| 3 | 把它做成工具 | 带说明的输入输出 + 测试 + 追踪 | agent 能调用；坏了马上知道 |
| 4 | 第一次考试 | dev-200 | 82%：好得可疑 |
| 5 | 查泄漏，遮住再考 | 考试时遮住病例自己的论文 | 真实的分数：49% |
| 6 | 选打分方法 | 在 dev-200 上比 3 种 | 用开发集做选择 |
| 7 | 三个基线放在一起 | MLflow 里的记录 | agent 要过的线 |

> **顺手修了一个 bug，单独提交（合并请求 #11）。** 做这一章时发现，MLflow 的"使用情况统计"其实一直没关掉。原因是导入的顺序：MLflow 先被加载，我们的设置后到，没赶上。修法是让 `raredx` 一被导入就先把设置写好（Python 保证一个包的 `__init__` 最先运行）。**它和这一章的内容无关，所以单独开了一个合并请求**：一个合并请求只做一件事，出了问题好查，也好单独退回。

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 工具要读的知识：HPO 官方数据

### 在干什么
下载 HPO（人类表型本体）官方发布的文件，写进数据源清单，版本和指纹都钉死。

### 背景
第 05 章钉住了**考题**的版本。工具读的**知识**也一样要钉：HPO 每一两个月发一版，疾病的注释会增删。不钉版本，同一个工具这个月 49%、下个月 52%，你分不清是工具变了，还是知识变了。

### 用了什么技术，为什么
HPO 每一版都发布在 GitHub 上。我们用其中 4 个文件：

| 文件 | 里面是什么 |
|---|---|
| `hp.obo` | 本体：约 2 万个症状术语，以及谁是谁"更具体的一种"（"感音神经性耳聋"是"听力障碍"的一种） |
| `phenotype.hpoa` | 疾病注释：每个病有哪些症状，**每一行都写明出处**（哪篇论文） |
| `genes_to_disease.txt` | 每个病的致病基因 |
| `genes_to_phenotype.txt` | 基因和症状的对应（hpo3 加载时需要） |

数据源清单里加了一段 `[hpo]`。一个数据源现在可以有好几个文件，**每个文件一个指纹**。这些指纹不是我们自己算的：GitHub 给每个发布文件都公布了官方的 sha256，直接抄过来。下载后对不上，文件当场删掉。

### 公司里什么情况怎么做
- **知识库也是数据**：和考题一样钉版本、核指纹；每次评测都记下用的是哪一版（这一章的评测都记了 `hpo_release`）
- **升级知识库**：改清单里的版本，在 dev-200 上重跑，和旧版比
- **能不能用**：HPO 可以免费用，要求注明出处和版本（写在清单的 `license` 里）

### 执行
看清单里新加的一段，再运行下载（已经下载过的只核对指纹，不重复下载）：

In [ ]:
registry_text = (ROOT / "evals/datasets.toml").read_text()
print(registry_text[registry_text.index("[hpo]"):])
run("uv run raredx data fetch hpo")

### 结果怎么看
- `[hpo.files]` 下面 4 个文件，每个一行官方指纹
- 4 行 `ok … sha256 verified`：文件在 `data/raw/hpo/v2026-09-01/`，和官方指纹一致

### 面试怎么说
> "The knowledge a tool reads is pinned like the evaluation data: the HPO release sits in the same registry, with a checksum per file taken from GitHub's published digests, and every run records which release it used."

---

## 第 2 步 · 用现成的库打分：hpo3

### 在干什么
用一个行业里常用的现成库，按症状给所有疾病打分。

### 背景
"按症状给病打分"是罕见病诊断里最成熟的做法，算法早就定型了（Phenomizer，2009 年）。旧项目里这部分是自己写的（`hpo_graph.py`，两百多行）：没人审过，没有测试，算错了也不知道。

### 用了什么技术，为什么
- 临床上最常用的整套工具是 **Exomiser**（英国十万基因组计划在用）。但它是一整条 Java 流水线，要病人的测序文件和几十 GB 的数据。我们现在只要它里面"按症状打分"这一块
- Python 里做这件事最常用的是 **PyHPO**。**hpo3** 是同一个作者用 Rust 重写的版本：用法一样，快得多。柏林 Charité 医院的临床变异分析平台 VarFish，底层用的就是同一个 Rust 库
- 许可证是 **MIT**，公司可以放心用。另一个候选 hpo-toolkit 是 **GPL**：GPL 要求用了它的软件在分发时也公开源代码，公司一般要先过法务

**选库时公司看的几样**：行业里有没有人在用、还有没有人维护（hpo3 最近一版是 2025 年 12 月）、许可证、能不能装上（Mac 和 CI 用的 Linux 都有现成的安装包）。

**打分的原理**，三条：
1. **越少见的症状越值钱**。"癫痫"上千个病都有，对上了说明不了什么；"晶状体脱位"只有几十个病有，对上了就很有分量。这个"少见程度"叫**信息量**
2. **相近的症状算部分对上**。病人写的是"感音神经性耳聋"，某个病的注释是"听力障碍"：在术语树上，前一个是后一个的下一级。两个症状的相似度，就是它们**共同的上级**的信息量（Resnik 方法）
3. **两个方向都看**。病人的每个症状，在这个病的症状里找最像的；这个病的每个症状，在病人身上找最像的；两边各取平均，再平均（funSimAvg）

> **用现成的库，也要看清它读进来的是什么。** 疾病注释里有 3,072 行写着"0/n"：查过 n 个病人，**一个都没有**这个症状。hpo3 自己读文件时，会把这些行当成"有"。所以疾病的症状列表我们自己从文件里读（几十行代码），把这些行去掉；术语树和打分交给 hpo3。

### 公司里什么情况怎么做
- **先找现成的**：成熟的算法有成熟的库。自己写的版本要多写测试、多人审，出了 bug 也没人帮你修
- **看许可证**：MIT、BSD、Apache 一般可以直接用；GPL 先问法务
- **钉版本**：库的版本记在 `uv.lock` 里（第 02 章），换版本要重跑评测

### 执行
加载 HPO，看两个症状各自的"少见程度"，再看两对症状的相似度：

In [ ]:
from pyhpo import Ontology
from raredx.data import sources
from raredx.knowledge import hpo as hpo_knowledge

registry = sources.load_sources(ROOT / "evals/datasets.toml")
hpo = hpo_knowledge.load(sources.raw_dir(registry["hpo"], ROOT / "data"))
print(hpo, "\n")

for term_id in ["HP:0001250", "HP:0001083"]:  # 癫痫，晶状体脱位
    term = Ontology.get_hpo_object(term_id)
    print(f"{term.name:<16} 有这个症状的病 {len(term.omim_diseases):>5} 个   信息量 {term.information_content['omim']:.2f}")

snhl, hearing, seizure = (Ontology.get_hpo_object(t) for t in ["HP:0000407", "HP:0000365", "HP:0001250"])
print(f"\n{snhl.name} vs {hearing.name}：相似度 {snhl.similarity_score(hearing, kind='omim', method='resnik'):.2f}")
print(f"{snhl.name} vs {seizure.name}：相似度 {snhl.similarity_score(seizure, kind='omim', method='resnik'):.2f}")

### 结果怎么看
- 第一行：这一版 HPO 里有注释的 OMIM 病，八千多个
- 癫痫：一千九百多个病都有，信息量 1.5；晶状体脱位：几十个病有，信息量 5.2。对上一个晶状体脱位，顶得上三个多癫痫
- 感音神经性耳聋 vs 听力障碍：1.90，正好是"听力障碍"本身的信息量（它就是两者共同的上级）；耳聋 vs 癫痫：0，两者毫不相干

### 面试怎么说
> "Phenotype matching uses the standard method — Resnik similarity over the HPO, with information content from the disease annotations, best-match averaged — through hpo3, an MIT-licensed Rust implementation of PyHPO. We build the disease profiles ourselves only to drop entries recorded as absent, which the library counts as present."

---

## 第 3 步 · 把它做成工具

### 在干什么
把"打分"包成一个**工具**：一个输入、输出都写清楚的函数，以后 agent 直接调用它。

### 背景
agent 里的大模型不读代码。它决定调不调一个工具、怎么调，只看三样东西：**工具的名字、工具的说明、每个参数的说明**。所以工具的说明是写给模型看的，和提示词一样重要。

### 用了什么技术，为什么
工具在 [`src/raredx/tools/rank_diseases.py`](../../src/raredx/tools/rank_diseases.py)：

| | 内容 |
|---|---|
| 名字 | `rank_diseases` |
| 说明 | 按病人的症状给所有 OMIM 病排序；只知道 HPO 注释过的；分数不是概率 |
| 输入 | `hpo_ids`：症状编号的列表（必须是 `HP:` 加 7 位数字）；`top_k`：要几个（1 到 50） |
| 输出 | 排好的病（编号、名字、致病基因、分数）；**不认识的编号单独列出来**；用的 HPO 版本 |

几个设计：
- **输入格式不对就报错**（pydantic 校验）：模型写错了格式，马上收到一条看得懂的错误，改了可以重试
- **不认识的编号不报错，而是列出来**：可能是别的版本的编号；模型需要知道哪些没用上
- **每次调用都记成追踪里的一层**，类型是 TOOL（工具）。第 09 章里调模型那一层叫 `Completions`；以后 agent 每调一次这个工具，追踪里就多一层 `rank_diseases`

**测试**：测试里不能用真的 HPO（CI 里没有数据，第 04 章）。所以做了一个**迷你 HPO**（`tests/fixtures/hpo/`）：13 个术语、4 个假的病，专门放进几种情况：有一行"0/n"、有一个过时的术语、有一个病只出自一篇论文、有一个病有两个致病基因。每种情况一条测试。

> 测试还守着一件事：MLflow 默认把记录写进**当前目录**的 `mlflow.db`。在项目里跑测试，就会写进真的评测记录。所以测试一开始就把 MLflow 指向一个临时目录。

### 公司里什么情况怎么做
- **工具的说明要审**：改工具说明和改提示词一样，走合并请求、重跑评测
- **工具单独测**：每个工具不靠 agent 就能测。agent 出错时，先排除是工具的问题
- **以后包成 MCP**：MCP 是现在通用的"工具接口"标准，同一个工具，各家的 agent 都能接上。我们的工具已经是"带说明的函数"，到时候外面包一层就行

### 执行
用第 05 章那道 Perrault 的题，调一次工具。随手的调用也会被记录，放在 MLflow 一个叫 `playground` 的实验里，不和正式评测混在一起：

In [ ]:
import mlflow
from raredx.config import settings
from raredx.data import cases as case_store
from raredx.tools.rank_diseases import RankDiseases

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment("playground")

by_id = {c.id: c for c in case_store.read(sources.processed_dir(registry["phenopacket-store"], ROOT / "data"))}
case = by_id["PMID_39701103_Proband_F3"]
rank_diseases = RankDiseases(hpo)

print("症状：", case.hpo_labels, "\n")
ranking = rank_diseases(case.hpo_ids, top_k=10)
for i, m in enumerate(ranking.matches, 1):
    mark = "   ← 正确答案" if m.disease_id == case.disease_id else ""
    print(f"{i:>2}. {m.disease_id}  {m.score:4.2f}  {', '.join(m.genes) or '-':<10} {m.disease_name[:50]}{mark}")
print("\n不认识的编号：", ranking.unrecognized or "无", "  HPO 版本：", ranking.hpo_release)

再跑一遍这个工具的测试：

In [ ]:
run("uv run pytest tests/knowledge tests/tools tests/baselines/test_phenotype_rank.py")

### 结果怎么看
- 正确答案 Perrault 7 型（OMIM:621101）排在**第 6**。前几名是各种遗传性耳聋：病人症状里耳聋最显眼
- 每个病带着它的致病基因（Perrault 7 型是 DAP3）
- 测试全部通过，几秒钟；不需要网络，也不需要真数据

**先记住这个"第 6 名"**，第 5 步还会回来看它。

### 面试怎么说
> "Each tool is a typed function whose name, description and parameter descriptions are written for the model; inputs are validated, unknown terms are reported rather than fatal, and every call is a TOOL span in the trace. The tests run on a miniature ontology built to hit each edge case."

---

## 第 4 步 · 第一次考试：好得可疑

### 在干什么
把这个工具当成一个"系统"来考：**B0 = 只用工具 T2，不用大模型**，在 dev-200 上考一次。

### 背景
B0 回答的是：**不用大模型，只按症状打分，能做到多好**。这是 agent 要过的一条线：agent 要是还不如它，用大模型就没有意义。

### 用了什么技术，为什么
B0 在 [`src/raredx/baselines/phenotype_rank.py`](../../src/raredx/baselines/phenotype_rank.py)，十几行：把题目里的症状交给工具，工具排出的前 10 个病就是答卷。考试用第 08 章那一套，一行命令。

这里先用**最直接的方式**考（命令最后那个开关是什么，第 5 步解释）。

### 公司里什么情况怎么做
- **结果好得出乎意料，先怀疑，再高兴**：简单的方法远远超过复杂的，常常不是方法好，而是评测有漏洞
- **有了数字，先和对照比**：瞎猜 9%，大模型 28%

### 执行

In [ ]:
run("uv run raredx eval run --system phenotype-rank --split dev-200 --no-mask-source-paper")

### 结果怎么看
- 前 10 名答对 **82.0%**，第 1 名就答对 50.5%
- 大模型是 28%。一个 2009 年的老方法，比大模型高出将近 3 倍，第一名就猜中一半？

**这不正常，要先查清楚，再相信它。**

### 面试怎么说
> "The phenotype baseline scored 82% top-10, nearly three times the model. That was the signal to look for a leak before believing it."

---

## 第 5 步 · 查泄漏，遮住再考

### 在干什么
查出分数为什么这么高，然后把评测修好。

### 背景
回想第 1 步：疾病注释的**每一行都写着出处**。HPO 整理疾病注释，用的是已发表的病例报告；Phenopacket 的考题**也是从病例报告整理出来的**，两边用的常常是同一批论文。也就是说，工具考某一道题时，它读的知识里，可能就有这道题自己的那篇论文。相当于开卷考试，书里还夹着答案。

### 用了什么技术，为什么
**先量一下有多严重**。200 道题里，有多少道题自己的论文，就是它那个病的注释出处？再看第 3 步那道 Perrault 题：

In [ ]:
from raredx.evals import splits

dev = [by_id[i] for i, _ in splits.read(ROOT / "evals/splits/dev-200.tsv")]
cited = sum(hpo.cites(c.disease_id, c.pmid) for c in dev)
print(f"dev-200 里，自己的论文就是疾病注释出处的题：{cited} / {len(dev)} 道\n")

print("Perrault 7 型在 HPO 里的注释出处：", sorted(hpo.references(case.disease_id)))
print("这道题来自的论文：                ", case.pmid)

**怎么修**：考某一道题时，把**这道题自己的那篇论文**从工具的知识里遮住，只出自这篇论文的注释，这道题就看不到。别的论文的知识照常用：真实的医生也查得到它们。

遮的办法是公司里常用的"**每次请求单独的设置**"（Python 的 `contextvars`）：评测框架出每一道题时，悄悄设一个"这道题要遮住哪篇论文"；工具查知识时看一眼这个设置。**系统和模型都看不到它，也不知道它存在。** 好处是：以后不管是 B0、固定流程还是 agent，只要用我们的工具，考试时都自动遮住。

现在这是**默认**：每次评测都遮，MLflow 里每次运行都记一个参数 `knowledge_mask = case source paper`。上一步用的 `--no-mask-source-paper` 就是把它关掉，只用来量泄漏有多大；这样跑的运行，名字后面会标 `(unmasked)`。

### 公司里什么情况怎么做
- **泄漏检查是评测的一部分**：每接一种新的知识来源（文献检索、数据库），先问"它会不会读到考题本身"
- **在评测框架里统一处理**：不靠每个工具自己记得遮。框架统一设、工具统一读，测一次处处有效
- **两个数都留着**：遮了的和没遮的都记在 MLflow 里，差多少就是泄漏有多大

### 执行
先看 Perrault 这道题遮住之后怎样，再遮住论文重考 200 道题（现在是默认，不用加开关）：

In [ ]:
from raredx.knowledge import masking

with masking.masked([case.pmid]):
    masked = rank_diseases(case.hpo_ids, top_k=10)
print("遮住这道题自己的论文以后，Perrault 7 型还在前 10 名里吗？",
      any(m.disease_id == case.disease_id for m in masked.matches))

In [ ]:
run("uv run raredx eval run --system phenotype-rank --split dev-200")

### 结果怎么看
- **153 / 200** 道题，自己的论文就是疾病注释的出处
- Perrault 7 型的注释**全部**来自一篇论文，就是这道题的那篇。遮住以后，HPO 里没有这个病的任何症状，工具找不到它。这其实正是真实情况：一个病刚报道出来的时候，知识库里还没有它
- 遮住后重考：前 10 名 **49.0%**（42.2–55.9），第 1 名 19.0%。比 82% 低了一大截，但仍然比大模型的 28% 高

还剩一点点没遮干净：hpo3 计算"少见程度"时用的是全部注释，没有逐题遮。影响很小，这里记下，暂不处理。

### 面试怎么说
> "HPO curates disease annotations from the same case reports Phenopacket Store is built from: for 153 of the 200 dev cases, the case's own paper was a source. The harness now masks each case's paper from the tools' knowledge, through a context variable the system never sees. The honest number is 49%, down from 82%."

---

## 第 6 步 · 用开发集选打分方法

### 在干什么
hpo3 有好几种打分方法。在 dev-200 上比一比，选最好的。

### 背景
第 06 章说过：dev-200 是用来**调参数、做选择**的，heldout 留到最后只考一次。这是第一次真的用 dev-200 做选择。

### 用了什么技术，为什么
比 3 种（都遮住论文）：

| 方法 | 是什么 |
|---|---|
| `graphic/funSimAvg` | hpo3 默认的方法 |
| `resnik/BMA` | Resnik，换一种合并方式 |
| `resnik/funSimAvg` | Resnik + 两个方向平均：最经典的组合（第 5 步那次就是它） |

每种都是一次正式评测，记在 MLflow 里，名字带着方法，比如 `phenotype-rank (graphic/funSimAvg)`。

### 公司里什么情况怎么做
- **每次尝试都留记录**：试过哪些、各多少分，都在 MLflow 里。以后有人问"为什么用这个"，翻记录就有答案
- **选项别太多**：200 道题的误差范围约 ±7%，试几十种，总有一种碰巧看起来好。只试有道理的几种
- **选定了写进代码**：默认值就是选出来的那个，代码里写明为什么

### 执行
每种十几秒：

In [ ]:
for similarity in ["graphic/funSimAvg", "resnik/BMA"]:
    run(f"uv run raredx eval run --system phenotype-rank --split dev-200 --similarity {similarity}")

### 结果怎么看

| 方法 | 前 1 名 | 前 10 名 |
|---|---|---|
| graphic/funSimAvg（库的默认） | 17.0% | 38.0% |
| resnik/BMA | 10.5% | 30.5% |
| **resnik/funSimAvg（经典组合）** | **19.0%** | **49.0%** |

经典组合最好，所以它是工具的默认。同一个工具，方法不同，前 10 名能差将近 20 个百分点：**库的默认值不一定适合你的问题**，要用自己的评测来选。

### 面试怎么说
> "The similarity measure was chosen on the dev set among a few standard options, each recorded as an MLflow run; the classic Resnik best-match average beat the library default by 11 points top-10."

---

## 第 7 步 · 三个基线放在一起

### 在干什么
把目前所有的系统放进一张表。agent 以后要和它们比。

### 执行

In [ ]:
run("uv run raredx eval runs --split dev-200")

### 结果怎么看
列表从上往下是最近的运行。整理一下：

| 系统 | 前 1 名 | 前 10 名 | 说明 |
|---|---|---|---|
| 瞎猜（popularity） | 0.5% | 9.0% | 每个病人都给同样的 10 个病 |
| 大模型直接猜（llm-direct） | 14.5% | 28.0% | 不给工具，一次回答 |
| **按症状打分（phenotype-rank，B0）** | **19.0%** | **49.0%** | 不用大模型，遮住论文 |
| 按症状打分，不遮（unmasked） | 50.5% | 82.0% | 有泄漏，**不能当成绩** |

这张表说明两件事：
1. **症状已经是标准编号时，老方法比大模型强。** 大模型的价值不在这里和老方法比赛，而在老方法做不到的事：读医生写的自由文字、查文献、讲出理由
2. **agent 不应该取代这个工具，而应该用它。** B0 是 agent 的起点：agent 拿到它的排名，再去查、去核对，目标是在 49% 之上继续往上走

还要记住：大模型的 28% 也可能有"泄漏"，模型训练时可能读过这些论文（第 09 章）。

### 面试怎么说
> "On coded phenotypes, classical similarity beats a single LLM call, 49% to 28% top-10, so the agent is designed to use it as a tool and improve on it, not to replace it."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 知识 | HPO 官方发布 + 数据源清单 + 官方指纹 | 知识和考题一样钉版本 |
| 2 打分 | hpo3（MIT）+ 自己读注释、去掉"0/n" | 用成熟的库，也看清它读了什么 |
| 3 工具 | 带说明的输入输出 + 迷你 HPO 测试 + TOOL 追踪 | 模型能调用，坏了马上知道 |
| 4 第一次考试 | dev-200 | 好得可疑，先怀疑 |
| 5 查泄漏 | 逐题遮住来源论文（`contextvars`） | 真实的分数：82% → 49% |
| 6 选方法 | dev-200 上比 3 种，记在 MLflow | 用评测做选择 |
| 7 对比 | 三个基线一张表 | agent 要过的线 |

**一句话**：第一个工具是"按症状给所有罕见病排序"。它不用大模型，前 10 名答对 49%，比大模型直接猜高；过程中查出并堵上了一个大泄漏（不堵是 82%）。

## 面试题

1. **Your simple baseline beat the LLM by a wide margin. What did you do?**
   > "Treated it as a red flag. The annotations it matches against are curated from the same papers as the test cases; I measured it — 153 of 200 dev cases — and masked each case's source paper during evaluation. The score fell from 82% to 49%, still above the LLM's 28%."

2. **How do you prevent leakage between a knowledge base and the test set?**
   > "Every knowledge entry keeps its source. The evaluation harness sets, per case, which sources are masked, through a context variable that the tools read and the system never sees, so any system that uses our tools is evaluated leak-free by default."

3. **How do you design a tool for an LLM agent?**
   > "A typed function with a name, a description and parameter descriptions written for the model; strict input validation with errors the model can act on; partial failures reported in the output rather than raised; tested on its own; traced as a TOOL span."

4. **Did you build or reuse the phenotype matching?**
   > "Reused: hpo3, an MIT-licensed Rust implementation of PyHPO, the same core a clinical variant platform runs on. But I checked what it reads: it counts annotations recorded as absent as present, so the disease profiles are built from the file directly."

5. **How did you choose the similarity measure?**
   > "On the dev set, among three standard options, each recorded as an MLflow run. The held-out set stays untouched until the design is frozen."

**下一章 11 · 第二个工具：查一个病的资料**：某个病有哪些典型症状、遗传方式、致病基因，出自哪几篇论文。agent 拿到 B0 的排名之后，要能一个一个去核对。